# TML Assignment 3 - Robustness
## Experiment 2: ResNet34 + TRADES (Zhang et al., 2019)

**Why this experiment**: our first submission (resnet18, PGD-AT / Madry) scored 0.5337 on the
leaderboard (locally: clean=0.643, PGD-20=0.433). For reference, published PGD-AT resnet18
baselines at eps=8/255 reach ~83-87% clean / ~42-45% robust - our robust accuracy is in the
right range, but clean accuracy has a lot of headroom. This points to two things to fix:

1. **Training instability**: the constant lr=0.1 phase caused large accuracy swings
   (epoch 18: clean 49% -> epoch 19: clean 34%). Fixed here with **LR warmup + cosine
   annealing** instead of step decay.
2. **PGD-AT trains only on adversarial examples**, which sacrifices more clean accuracy
   than necessary. **TRADES** (Zhang et al., 2019 - not in the assignment's reference
   list, but the natural extension of Madry's framework) adds back a clean CE loss term
   plus a *tunable* KL-divergence robustness regularizer (`beta`), explicitly trading off
   clean vs. robust accuracy - well suited to our 0.5*clean + 0.5*robust score.

Also switching to **resnet34** for more capacity (next architecture step per Madry's
finding that capacity matters for fitting the robust objective).

**TRADES objective**:
```
loss = CE(f(x), y) + beta * KL( f(x_adv) || f(x) )
```
where `x_adv` is found via PGD maximizing the KL divergence (not the CE loss) between the
adversarial and clean output distributions. `beta=6` is the paper's standard CIFAR-10
setting for eps=8/255 (favors robustness; lower beta favors clean accuracy).

Same constraints as before: inputs in `[0,1]`, no external normalization, plain
`torchvision.models.resnet34` with `fc` replaced for 9 classes.

Run cells top to bottom (Runtime -> Run all).

## 1. Setup

In [ ]:
import torch
print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os
PROJECT_DIR = '/content/drive/MyDrive/tml_assignment3'
CKPT_DIR = os.path.join(PROJECT_DIR, 'checkpoints')
os.makedirs(CKPT_DIR, exist_ok=True)
print('Checkpoint dir:', CKPT_DIR)

## 2. Download dataset

In [ ]:
DATA_PATH = '/content/train.npz'
if not os.path.exists(DATA_PATH):
    !wget -q -O {DATA_PATH} "https://huggingface.co/datasets/SprintML/tml26_task3/resolve/main/train.npz"
print('Downloaded:', os.path.exists(DATA_PATH), os.path.getsize(DATA_PATH) / 1e6, 'MB')

## 3. Imports & hyperparameters

In [ ]:
import time
import math
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, random_split
from torchvision.models import resnet34
import torchvision.transforms.v2 as T
from tqdm.auto import tqdm

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True

# ---- Hyperparameters ----
NUM_CLASSES = 9
MODEL_NAME = 'resnet34'
BATCH_SIZE = 128
EPOCHS = 50
WARMUP_EPOCHS = 3
VAL_SIZE = 5000
SEED = 42

# Optimizer / schedule: linear warmup -> cosine annealing (replaces the unstable
# constant-lr-then-step-decay schedule from experiment 1)
LR = 0.1
MOMENTUM = 0.9
WEIGHT_DECAY = 5e-4

# TRADES
EPS = 8 / 255
ALPHA = 2 / 255
PGD_STEPS_TRAIN = 7
TRADES_BETA = 6.0

# PGD attack (evaluation - stronger)
PGD_STEPS_EVAL = 20

USE_AMP = (device.type == 'cuda')

RESUME = True
CKPT_PATH = os.path.join(CKPT_DIR, f'{MODEL_NAME}_trades.pt')

torch.manual_seed(SEED)
np.random.seed(SEED)
print('Device:', device, '| AMP:', USE_AMP)

## 4. Data loading

In [ ]:
data = np.load(DATA_PATH)
images = torch.from_numpy(data['images']).float() / 255.0  # (N, 3, 32, 32) in [0,1]
labels = torch.from_numpy(data['labels']).long()
print('Images:', images.shape, 'Labels:', labels.shape)

full_dataset = TensorDataset(images, labels)
train_size = len(full_dataset) - VAL_SIZE
train_set, val_set = random_split(
    full_dataset, [train_size, VAL_SIZE],
    generator=torch.Generator().manual_seed(SEED)
)
print('Train size:', len(train_set), 'Val size:', len(val_set))

augment = T.Compose([
    T.RandomCrop(32, padding=4, padding_mode='reflect'),
    T.RandomHorizontalFlip(),
])

# num_workers=0: data is already an in-memory TensorDataset
train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, drop_last=True)
val_loader = DataLoader(val_set, batch_size=256, shuffle=False, num_workers=0)

## 5. Model

In [ ]:
def build_model():
    model = resnet34(weights=None)
    model.fc = nn.Linear(model.fc.in_features, NUM_CLASSES)
    return model

model = build_model().to(device)

model.eval()
with torch.no_grad():
    out = model(torch.randn(1, 3, 32, 32, device=device))
assert out.shape == (1, NUM_CLASSES), out.shape
print('Output shape OK:', out.shape)

## 6. Attacks (TRADES inner maximization, PGD/FGSM for eval)

In [ ]:
def trades_inner_max(model, x, eps, alpha, steps):
    """Find x_adv maximizing KL(f(x_adv) || f(x)) - the TRADES inner maximization.
    Model is set to eval() during this so BatchNorm uses running statistics."""
    was_training = model.training
    model.eval()

    x_natural = x.detach()
    with torch.no_grad(), torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
        p_natural = F.softmax(model(x_natural), dim=1)

    x_adv = x_natural + 0.001 * torch.randn_like(x_natural)
    x_adv = torch.clamp(x_adv, 0.0, 1.0)

    for _ in range(steps):
        x_adv.requires_grad_(True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            log_p_adv = F.log_softmax(model(x_adv), dim=1)
            loss_kl = F.kl_div(log_p_adv, p_natural, reduction='batchmean')
        grad = torch.autograd.grad(loss_kl, x_adv)[0]
        with torch.no_grad():
            x_adv = x_adv + alpha * grad.sign()
            x_adv = torch.min(torch.max(x_adv, x_natural - eps), x_natural + eps)
            x_adv = torch.clamp(x_adv, 0.0, 1.0)

    if was_training:
        model.train()
    return x_adv.detach()


def pgd_attack(model, x, y, eps, alpha, steps, random_start=True):
    """Standard untargeted L_inf PGD (CE-based) - used for evaluation."""
    was_training = model.training
    model.eval()

    x_orig = x.detach()
    if random_start:
        delta = torch.empty_like(x_orig).uniform_(-eps, eps)
        x_adv = torch.clamp(x_orig + delta, 0.0, 1.0).detach()
    else:
        x_adv = x_orig.clone()

    for _ in range(steps):
        x_adv.requires_grad_(True)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            logits = model(x_adv)
            loss = F.cross_entropy(logits, y)
        grad = torch.autograd.grad(loss, x_adv)[0]
        with torch.no_grad():
            x_adv = x_adv + alpha * grad.sign()
            x_adv = torch.clamp(x_adv, x_orig - eps, x_orig + eps)
            x_adv = torch.clamp(x_adv, 0.0, 1.0)

    if was_training:
        model.train()
    return x_adv.detach()


def fgsm_attack(model, x, y, eps):
    was_training = model.training
    model.eval()
    x_orig = x.detach().requires_grad_(True)
    with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
        logits = model(x_orig)
        loss = F.cross_entropy(logits, y)
    grad = torch.autograd.grad(loss, x_orig)[0]
    x_adv = torch.clamp(x_orig + eps * grad.sign(), 0.0, 1.0).detach()
    if was_training:
        model.train()
    return x_adv

## 7. Optimizer, LR schedule (warmup + cosine), training loop

In [ ]:
optimizer = torch.optim.SGD(model.parameters(), lr=LR, momentum=MOMENTUM, weight_decay=WEIGHT_DECAY)
scaler = torch.amp.GradScaler(device.type, enabled=USE_AMP)


def lr_lambda(epoch):
    if epoch < WARMUP_EPOCHS:
        return (epoch + 1) / WARMUP_EPOCHS
    progress = (epoch - WARMUP_EPOCHS) / max(1, EPOCHS - WARMUP_EPOCHS)
    return 0.5 * (1 + math.cos(math.pi * progress))


scheduler = torch.optim.lr_scheduler.LambdaLR(optimizer, lr_lambda=lr_lambda)

start_epoch = 0
if RESUME and os.path.exists(CKPT_PATH):
    ckpt = torch.load(CKPT_PATH, map_location=device)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    if 'scaler_state_dict' in ckpt:
        scaler.load_state_dict(ckpt['scaler_state_dict'])
    start_epoch = ckpt['epoch'] + 1
    print(f'Resumed from epoch {start_epoch}')
else:
    print('Starting fresh training')

In [ ]:
@torch.no_grad()
def evaluate_clean(model, loader):
    model.eval()
    correct, total = 0, 0
    for x, y in loader:
        x, y = x.to(device), y.to(device)
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            pred = model(x).argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)
    return correct / total


def evaluate_pgd(model, loader, eps, alpha, steps, max_batches=None):
    model.eval()
    correct, total = 0, 0
    for i, (x, y) in enumerate(loader):
        if max_batches is not None and i >= max_batches:
            break
        x, y = x.to(device), y.to(device)
        x_adv = pgd_attack(model, x, y, eps, alpha, steps)
        with torch.no_grad(), torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            pred = model(x_adv).argmax(dim=1)
        correct += (pred == y).sum().item()
        total += y.size(0)
    return correct / total

In [ ]:
best_score = -1.0
BEST_CKPT_PATH = os.path.join(CKPT_DIR, f'{MODEL_NAME}_trades_best.pt')
if RESUME and os.path.exists(BEST_CKPT_PATH):
    best_ckpt = torch.load(BEST_CKPT_PATH, map_location='cpu')
    best_score = best_ckpt.get('score', -1.0)
    print(f'Loaded best score so far: {best_score:.4f}')

for epoch in range(start_epoch, EPOCHS):
    model.train()
    t0 = time.time()
    running_loss, running_correct, running_total = 0.0, 0, 0

    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1}/{EPOCHS}', leave=False)
    for x, y in pbar:
        x, y = x.to(device), y.to(device)
        x = augment(x)

        x_adv = trades_inner_max(model, x, EPS, ALPHA, PGD_STEPS_TRAIN)

        model.train()
        optimizer.zero_grad()
        with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=USE_AMP):
            logits_natural = model(x)
            logits_adv = model(x_adv)
            loss_natural = F.cross_entropy(logits_natural, y)
            loss_robust = F.kl_div(
                F.log_softmax(logits_adv, dim=1),
                F.softmax(logits_natural, dim=1),
                reduction='batchmean',
            )
            loss = loss_natural + TRADES_BETA * loss_robust
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * y.size(0)
        running_correct += (logits_natural.argmax(dim=1) == y).sum().item()
        running_total += y.size(0)

        pbar.set_postfix(loss=running_loss / running_total, clean_acc=running_correct / running_total)

    scheduler.step()

    train_loss = running_loss / running_total
    train_clean_acc = running_correct / running_total
    val_clean_acc = evaluate_clean(model, val_loader)
    val_pgd_acc = evaluate_pgd(model, val_loader, EPS, ALPHA, PGD_STEPS_TRAIN, max_batches=5)
    val_score = 0.5 * val_clean_acc + 0.5 * val_pgd_acc

    dt = time.time() - t0
    print(f'Epoch {epoch+1}/{EPOCHS} | lr {scheduler.get_last_lr()[0]:.4f} | '
          f'train_loss {train_loss:.4f} | train_clean_acc {train_clean_acc:.4f} | '
          f'val_clean_acc {val_clean_acc:.4f} | val_pgd7_acc {val_pgd_acc:.4f} | '
          f'val_score {val_score:.4f} | {dt:.1f}s')

    ckpt = {
        'epoch': epoch,
        'model_state_dict': model.state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict': scaler.state_dict(),
        'val_clean_acc': val_clean_acc,
        'val_pgd7_acc': val_pgd_acc,
        'score': val_score,
    }
    torch.save(ckpt, CKPT_PATH)

    # Track the best checkpoint by val_score (mitigates robust overfitting -
    # Rice et al., 2020 - where late-epoch robust accuracy can degrade)
    if val_score > best_score:
        best_score = val_score
        torch.save(ckpt, BEST_CKPT_PATH)
        print(f'  -> new best (val_score={val_score:.4f}), saved to {BEST_CKPT_PATH}')

## 8. Final evaluation (best checkpoint, broadened attack suite)

In [ ]:
# Load the best checkpoint by val_score (not necessarily the last epoch)
best_ckpt = torch.load(BEST_CKPT_PATH, map_location=device)
model.load_state_dict(best_ckpt['model_state_dict'])
print(f"Loaded best checkpoint from epoch {best_ckpt['epoch']+1} (val_score={best_ckpt['score']:.4f})")

final_clean_acc = evaluate_clean(model, val_loader)
print(f'Clean accuracy: {final_clean_acc:.4f}')

# PGD-20 at multiple eps to see the robustness curve, plus 2 random restarts at eps=8/255
for eps_test in [4/255, 8/255, 16/255]:
    acc = evaluate_pgd(model, val_loader, eps_test, eps_test / 4, PGD_STEPS_EVAL)
    print(f'PGD-20 robust accuracy (eps={eps_test:.4f}): {acc:.4f}')

restart_accs = []
for r in range(2):
    torch.manual_seed(SEED + r)
    acc = evaluate_pgd(model, val_loader, EPS, ALPHA, PGD_STEPS_EVAL)
    restart_accs.append(acc)
    print(f'PGD-20 robust accuracy (eps=8/255, restart {r}): {acc:.4f}')
worst_pgd20 = min(restart_accs)

# FGSM
model.eval()
fgsm_correct, fgsm_total = 0, 0
for x, y in val_loader:
    x, y = x.to(device), y.to(device)
    x_adv = fgsm_attack(model, x, y, EPS)
    with torch.no_grad():
        pred = model(x_adv).argmax(dim=1)
    fgsm_correct += (pred == y).sum().item()
    fgsm_total += y.size(0)
final_fgsm_acc = fgsm_correct / fgsm_total
print(f'FGSM accuracy (eps=8/255): {final_fgsm_acc:.4f}')

print(f'\nEstimated score (0.5*clean + 0.5*worst PGD-20 @ eps=8/255): '
      f'{0.5*final_clean_acc + 0.5*worst_pgd20:.4f}')

## 9. Save submission state dict

In [ ]:
SUBMIT_PATH = os.path.join(PROJECT_DIR, f'{MODEL_NAME}_trades_submission.pt')

model.eval()
with torch.no_grad():
    out = model(torch.randn(1, 3, 32, 32, device=device))
assert out.shape == (1, NUM_CLASSES)

torch.save(model.state_dict(), SUBMIT_PATH)
print('Saved submission state dict to:', SUBMIT_PATH)
print('Model name for submission.py:', MODEL_NAME)